# Sentiment & Emotion - Top Engagement Dataset


In [1]:
import torch, time, os, gc
import json, re
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForSequenceClassification

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
if device.type == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"Start: {time.strftime('%Y-%m-%d %H:%M:%S')}")
SESSION_START = time.time()

CLEAN = Path('clean')

/home/defeo.k/dissertation-hpc-venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cuda
GPU: Tesla T4
Memory: 15.6 GB
Start: 2026-08-17 14:05:35


In [2]:
bsky = pd.read_csv(CLEAN / 'bsky_top_engagement.csv')
truth = pd.read_csv(CLEAN / 'truth_posts_clean.csv', dtype={'post_id': str})
print(f"Bluesky (top engagement): {len(bsky):,} posts")
print(f"Truth Social (full):      {len(truth):,} posts")

Bluesky (top engagement): 76,557 posts
Truth Social (full):      48,086 posts


## Model Registry & Functions

In [3]:
MODEL_REGISTRY = {
    'cardiffnlp/twitter-roberta-base-sentiment-latest': {
        'task': 'sentiment',
        'classification': 'single',
        'labels': {0: 'negative', 1: 'neutral', 2: 'positive'},
    },
    'cardiffnlp/twitter-roberta-base-emotion-multilabel-latest': {
        'task': 'emotion',
        'classification': 'multilabel',
        'labels': {0: 'anger', 1: 'anticipation', 2: 'disgust', 3: 'fear',
                   4: 'joy', 5: 'love', 6: 'optimism', 7: 'pessimism',
                   8: 'sadness', 9: 'surprise', 10: 'trust'},
    },
}

def preprocess(text):
    tokens = []
    for t in str(text).split():
        t = '@user' if t.startswith('@') and len(t) > 1 else t
        t = 'http' if t.startswith('http') else t
        tokens.append(t)
    return ' '.join(tokens)

def run_batch(texts, model, tokenizer, model_config, threshold=0.5):
    encoded = tokenizer(
        texts, padding=True, truncation=True, max_length=512, return_tensors='pt'
    ).to(device)
    with torch.no_grad(), torch.amp.autocast('cuda'):
        outputs = model(**encoded)
        logits = outputs.logits
    if model_config['classification'] == 'single':
        preds = torch.argmax(logits, dim=1).cpu().numpy()
        labels = model_config['labels']
        return [labels[p] for p in preds]
    else:
        probs = torch.sigmoid(logits).cpu().numpy()
        labels = model_config['labels']
        results = []
        for row in probs:
            active = [labels[i] for i, score in enumerate(row) if score >= threshold]
            results.append(active if active else ['neutral'])
        return results

def classify_fast(df, id_col, model_name, model_config, batch_size=256, threshold=0.5):
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSequenceClassification.from_pretrained(
        model_name, torch_dtype=torch.float16
    ).to(device)
    model.eval()

    texts = [preprocess(t) for t in df['text'].tolist()]
    all_results = []

    for i in tqdm(range(0, len(texts), batch_size), desc=model_name.split('/')[-1]):
        batch = texts[i:i+batch_size]
        preds = run_batch(batch, model, tokenizer, model_config, threshold)
        all_results.extend(preds)

    del model, tokenizer
    gc.collect()
    torch.cuda.empty_cache()

    if model_config['classification'] == 'single':
        return pd.DataFrame({id_col: df[id_col].values, 'label': all_results})
    else:
        return pd.DataFrame({id_col: df[id_col].values, 'labels': ['|'.join(r) for r in all_results]})

## Bluesky Sentiment

In [4]:

print("BLUESKY TOP ENGAGEMENT — SENTIMENT")
print(f"Started: {time.strftime('%H:%M:%S')}")
t0 = time.time()

bsky_sent = classify_fast(
    bsky, 'post_uri',
    'cardiffnlp/twitter-roberta-base-sentiment-latest',
    MODEL_REGISTRY['cardiffnlp/twitter-roberta-base-sentiment-latest']
)
bsky_sent.to_csv(CLEAN / 'bsky_top_sentiment.csv', index=False)
print(f"Done in {(time.time()-t0)/60:.1f} min")
print(bsky_sent['label'].value_counts().to_dict())

BLUESKY TOP ENGAGEMENT — SENTIMENT
Started: 14:05:36


/home/defeo.k/dissertation-hpc-venv/lib/python3.12/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
Some weights of the model checkpoint at cardiffnlp/twitter-roberta-base-sentiment-latest were not used when initializing RobertaForSequenceClassification: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model that yo

Done in 4.8 min
{'negative': 36646, 'neutral': 28468, 'positive': 11443}


## Bluesky Emotion

In [5]:

print("BLUESKY TOP ENGAGEMENT — EMOTION")
print(f"Started: {time.strftime('%H:%M:%S')}")
t0 = time.time()

bsky_emo = classify_fast(
    bsky, 'post_uri',
    'cardiffnlp/twitter-roberta-base-emotion-multilabel-latest',
    MODEL_REGISTRY['cardiffnlp/twitter-roberta-base-emotion-multilabel-latest']
)
bsky_emo.to_csv(CLEAN / 'bsky_top_emotions.csv', index=False)
print(f"Done in {(time.time()-t0)/60:.1f} min")

counts = Counter(e for labels in bsky_emo['labels'] for e in labels.split('|'))
for emotion, count in counts.most_common():
    print(f"  {emotion}: {count:,}")

BLUESKY TOP ENGAGEMENT — EMOTION
Started: 14:10:24


/home/defeo.k/dissertation-hpc-venv/lib/python3.12/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
twitter-roberta-base-emotion-multilabel-latest: 100%|██████████| 300/300 [04:39<00:00,  1.07it/s]


Done in 4.7 min
  disgust: 34,022
  anger: 33,339
  joy: 25,318
  anticipation: 21,450
  optimism: 16,939
  sadness: 6,551
  neutral: 5,114
  fear: 3,245
  love: 1,564
  surprise: 923
  pessimism: 562
  trust: 11


## Truth Social Sentiment

In [6]:

print("TRUTH SOCIAL — SENTIMENT")
print(f"Started: {time.strftime('%H:%M:%S')}")
t0 = time.time()

truth_sent = classify_fast(
    truth, 'post_id',
    'cardiffnlp/twitter-roberta-base-sentiment-latest',
    MODEL_REGISTRY['cardiffnlp/twitter-roberta-base-sentiment-latest']
)
truth_sent.to_csv(CLEAN / 'truth_top_sentiment.csv', index=False)
print(f"Done in {(time.time()-t0)/60:.1f} min")
print(truth_sent['label'].value_counts().to_dict())

TRUTH SOCIAL — SENTIMENT
Started: 14:15:07


/home/defeo.k/dissertation-hpc-venv/lib/python3.12/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
Some weights of the model checkpoint at cardiffnlp/twitter-roberta-base-sentiment-latest were not used when initializing RobertaForSequenceClassification: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model that yo

Done in 7.1 min
{'neutral': 23456, 'negative': 15616, 'positive': 9014}


## Truth Social Emotion

In [7]:

print("TRUTH SOCIAL — EMOTION")
print(f"Started: {time.strftime('%H:%M:%S')}")
t0 = time.time()

truth_emo = classify_fast(
    truth, 'post_id',
    'cardiffnlp/twitter-roberta-base-emotion-multilabel-latest',
    MODEL_REGISTRY['cardiffnlp/twitter-roberta-base-emotion-multilabel-latest']
)
truth_emo.to_csv(CLEAN / 'truth_top_emotions.csv', index=False)
print(f"Done in {(time.time()-t0)/60:.1f} min")

counts = Counter(e for labels in truth_emo['labels'] for e in labels.split('|'))
for emotion, count in counts.most_common():
    print(f"  {emotion}: {count:,}")

TRUTH SOCIAL — EMOTION
Started: 14:22:13


/home/defeo.k/dissertation-hpc-venv/lib/python3.12/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
twitter-roberta-base-emotion-multilabel-latest: 100%|██████████| 188/188 [07:03<00:00,  2.25s/it]


Done in 7.1 min
  anger: 22,901
  disgust: 22,758
  anticipation: 14,973
  joy: 14,494
  optimism: 13,973
  fear: 7,367
  neutral: 2,624
  sadness: 1,181
  love: 270
  surprise: 195
  pessimism: 92
  trust: 25


## Session Summary

In [8]:
elapsed = (time.time() - SESSION_START) / 3600

print("SESSION SUMMARY")
print(f"  Hardware: {torch.cuda.get_device_name(0)}")
print(f"  Bluesky top engagement: {len(bsky):,} posts")
print(f"  Truth Social: {len(truth):,} posts")
print(f"  Total time: {elapsed:.2f} hours")
print(f"  End: {time.strftime('%Y-%m-%d %H:%M:%S')}")
print()
for f in sorted(CLEAN.glob('*.csv')):
    print(f"  {f.name}: {f.stat().st_size/1e6:.1f} MB")

SESSION SUMMARY
  Hardware: Tesla T4
  Bluesky top engagement: 76,557 posts
  Truth Social: 48,086 posts
  Total time: 0.40 hours
  End: 2026-08-17 14:29:19

  bsky_comments_clean.csv: 619.4 MB
  bsky_posts_clean.csv: 903.6 MB
  bsky_reposts_clean.csv: 922.3 MB
  bsky_top_emotions.csv: 6.6 MB
  bsky_top_engagement.csv: 45.6 MB
  bsky_top_sentiment.csv: 6.1 MB
  bsky_top_topics.csv: 5.6 MB
  truth_comments_clean.csv: 1.5 MB
  truth_posts_clean.csv: 45.6 MB
  truth_reposts_clean.csv: 14.8 MB
  truth_top_emotions.csv: 1.7 MB
  truth_top_sentiment.csv: 1.3 MB
  truth_top_topics.csv: 1.0 MB
